# 1. Information about the submission


## 1.1 Name and number of the assignment


Final project. Hallucination Detection in Tool Calling


## 1.2 Student name


TODO: fill in student name before submission. Solo submission.


## 1.3 Codalab user ID / nickname / username


No public CodaLab leaderboard was specified for this final project. Repository: https://github.com/marritau/LLMs_final_project


## 1.4 Additional comments


This notebook is intentionally a facade over a public GitHub repository. The course chat allowed repository-based submissions when the project has an elaborated file structure. The repository can be installed with `pip install git+https://github.com/marritau/LLMs_final_project.git`, and the notebook calls the public facade functions from `tool_hallucination_detection`.


# 2. Technical Report


## 2.1 Methodology


The project builds a span-level hallucination detection benchmark for tool-calling dialogues. The base data source is ToolACE, which contains user queries, available tool descriptions, assistant tool calls, tool outputs, and final assistant answers. Each dialogue is normalized into a RAGTruth-style record with `query`, `context` as the tool output, and `output` as the final answer. The clean answer is used as a negative example, and three automatic corruption procedures produce positive examples with character-level span labels.

The three hallucination types follow the assignment statement. `tool_contradiction` replaces a value supported by the tool output with a conflicting value in the final answer. `overgeneration` appends plausible but unsupported information. `missing_tool` adds a recommendation that would require a tool absent from the available tool list. All labels are stored as spans with exact character offsets in the final answer, so the dataset supports both sentence-level binary evaluation and span-level detection.

The baseline suite contains an always-clean sanity baseline, a keyword/tool-action heuristic, an optional NLI sentence verifier, an optional LettuceDetect wrapper, and a LookBackLens-style support baseline. The full neural model is a token-classification encoder that receives `Question`, `Context`, and `Answer`; labels are applied only to answer tokens, while question and context tokens are ignored. For Colab-scale experiments the repository exposes a ModernBERT token-classifier training function, and the default quick path uses a fitted lightweight ensemble to keep the notebook reproducible without downloading large models.

The primary optimization target is sentence-level binary F1 because the course chat indicated that sentence-based ranking is the most likely evaluation signal. Span-level character F1, exact span F1, and relaxed IoU-based span F1 are still reported because the task explicitly asks for span-based labels.


## 2.2 Discussion of results


The quick run below is a smoke test on a tiny synthetic ToolACE-like sample; it verifies the full pipeline, offset handling, and metric computation. The full experiment should be run with `quick=False` on Colab/GPU to download ToolACE and optional model baselines. The report table produced by the code compares the sanity baselines, LookBackLens-style scoring, and the improved ensemble. For the final write-up, replace or augment the quick table with the full run table and include per-hallucination-type analysis for contradiction, overgeneration, and missing-tool cases.


# 3. Code


## 3.1 Requirements


In [ ]:
# In Colab or a clean environment, install from the public repository.
# If this notebook is run from inside the cloned repository, the fallback editable install is used.
import importlib.util
from pathlib import Path

if importlib.util.find_spec('tool_hallucination_detection') is None:
    repo_root = Path.cwd()
    if (repo_root / 'pyproject.toml').exists() and (repo_root / 'src').exists():
        %pip -q install -e .
    else:
        %pip -q install git+https://github.com/marritau/LLMs_final_project.git


## 3.2 Download and prepare the data


In [ ]:
from tool_hallucination_detection import prepare_dataset

# quick=True uses a tiny offline sample. Use quick=False for the full ToolACE-based run.
dataset = prepare_dataset(quick=True, cache_dir=None)
{split: len(records) for split, records in dataset.items()}


## 3.3 Preprocessing and label validation


In [ ]:
from tool_hallucination_detection.schema import validate_labels

for split, records in dataset.items():
    for record in records:
        validate_labels(record)

example = dataset['train'][0]
example


## 3.4 Experiments


In [ ]:
from tool_hallucination_detection import evaluate_experiment

result = evaluate_experiment(quick=True)
result['all_metrics']


### Sentence-level metrics


In [ ]:
result['sentence_metrics']


### Span-level metrics


In [ ]:
result['span_metrics']


### Per-type analysis


In [ ]:
result['per_type_metrics']


## 3.5 Full GPU run


In [ ]:
# Run this cell in Colab/GPU for the final numbers.
# It downloads ToolACE and attempts optional model-based baselines/training.
# full_result = evaluate_experiment(quick=False)
# full_result['all_metrics']
